# LLM Engineering: Deep Dive into Transformers

I am exploring the lower-level API of the `transformers` library to understand how models wrap PyTorch code. My goal is to run these experiments on a T4 GPU and build intuition for model architecture and inference pipelines.

In [ ]:
# Core dependencies for memory-efficient LLM serving
# bitsandbytes: 4-bit quantization
# accelerate: Device mapping and pipeline optimization
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [ ]:
# Authenticating with Hugging Face to access gated models like Llama
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Registering model checkpoints for testing
# Selecting Llama 3.1 8B for quantization tests and smaller models for raw inference
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

In [ ]:
# Implementing 4-bit Normal Float (NF4) quantization
# This allows loading an 8B parameter model (typically ~16GB) onto a 16GB T4 GPU
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Setup the tokenizer and prepare the chat input for the model
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
# Apply the chat template to convert the dictionary message into model-readable tokens
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
inputs

In [ ]:
# Instantiating the model with auto-device mapping and my quantization config
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

## Architectural Intuition

By printing the model object, I can inspect the PyTorch layers that make up the Transformer block:

- **Embeddings**: The lookup table mapping tokens to vectors.
- **Decoder Layers**: The repeating stack of Attention and Feed-Forward (MLP) layers.
- **LM Head**: The final projection layer that determines the next token probability.
- **Quantization Verification**: Checking if linear layers are replaced by `Params4bit` or similar wrappers.

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers
model

### Exploring Source Implementations

To understand the math behind the code, I can reference the official `transformers` source code. For example, the Llama implementation reveals how RoPE (Rotary Positional Embeddings) and KV caching are handled at the tensor level.

In [ ]:
# OK, with that, now let's run the model!
outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
# Tokenizer output is raw IDs; I need to decode them back to human-readable strings
tokenizer.decode(outputs[0])

In [ ]:
# Memory Management: Crucial for iterative development on limited VRAM
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## Inference Engineering: Streaming & Templates

- **TextStreamer**: Used for real-time token rendering to improve UX.
- **Chat Templates**: Ensures the model receives tokens in the specific format it was trained on (e.g., `<|begin_of_text|>` for Llama).

In [ ]:
# A reusable inference pipeline for testing different architectures and quantization states
def generate(model_name, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  tokenizer.pad_token = tokenizer.eos_token

  # Convert chat dicts to model-specific format
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")

  streamer = TextStreamer(tokenizer)

  if quant:
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)